# KinderCompass data preparation

This notebook demonstrates the supported offline data pipeline. Production logic lives in `SystemCode/src/scripts/prepare_data.py`; the notebook calls that module so CLI and interactive runs remain consistent.

In [ ]:
import sys
from pathlib import Path

start_dir = Path.cwd().resolve()
repo_root = next((path for path in (start_dir, *start_dir.parents) if (path / 'SystemCode').is_dir()), None)
if repo_root is None:
    raise RuntimeError('Could not locate the KinderCompass repository root')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from SystemCode.src.scripts.prepare_data import DEFAULT_OUTPUT, DEFAULT_RAW_DIR, prepare_catalogue, write_catalogue

## Prepare and inspect

This reads all required raw CSV and GeoJSON inputs, validates identifiers, joins location/licence/service evidence, and derives the processed catalogue in memory.

In [ ]:
catalogue = prepare_catalogue(DEFAULT_RAW_DIR)
print(f'Prepared {len(catalogue):,} schools')
catalogue[['school_id', 'centre_name_x', 'postal_code', 'town', 'base_fee', 'care_levels']].head()

In [ ]:
coverage = {
    'unique_school_ids': catalogue['school_id'].is_unique,
    'with_location': int(catalogue['has_location'].sum()),
    'with_fee_data': int(catalogue['has_fee_data'].sum()),
    'with_licence_data': int(catalogue['has_licence_data'].sum()),
    'with_vacancy_data': int(catalogue['has_vacancy_data'].sum()),
}
coverage

## Write the processed catalogue

The writer uses an atomic replacement so a failed write does not leave a partial master file. Running the next cell intentionally replaces the generated processed catalogue.

In [ ]:
destination = write_catalogue(catalogue, DEFAULT_OUTPUT)
print(f'Catalogue written to {destination}')